In [0]:
CATALOG = "workspace"
SCHEMA = "incident_assistent"
bronze_table = f"{CATALOG}.{SCHEMA}.bronze_docs"
CHUNK_SCHEMA = "chunk_id string, doc_id string, chunk_no int, text string, doc_hash string"

In [0]:
def chunk_text(text, chunk_size, overlap):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += chunk_size - overlap
    return chunks

def make_rows(row, chunk_size, overlap):
    return [(f"{row['doc_id']}_{i}", row["doc_id"], i, c, row["doc_hash"])
            for i, c in enumerate(chunk_text(row["text"], chunk_size, overlap))]

def build_chunk_table(chunk_size, overlap, table_name):
    rows = []
    for r in spark.table(bronze_table).collect():
        rows += make_rows(r, chunk_size, overlap)
    spark.createDataFrame(rows, CHUNK_SCHEMA).write.mode("overwrite").saveAsTable(table_name)
    spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(table_name, "->", len(rows), "chunks")

In [0]:
TABLE_A = f"{CATALOG}.{SCHEMA}.chunks_small"
TABLE_B = f"{CATALOG}.{SCHEMA}.chunks_large"

build_chunk_table(80, 15, TABLE_A)     # small chunks: 80 words, overlap 15
build_chunk_table(200, 40, TABLE_B)    # large chunks: 200 words, overlap 40

workspace.incident_assistent.chunks_small -> 63 chunks
workspace.incident_assistent.chunks_large -> 30 chunks


In [0]:
display(spark.table(TABLE_A).limit(5))

chunk_id,doc_id,chunk_no,text,doc_hash
adf_pipeline_failed_activity_0,adf_pipeline_failed_activity,0,"Azure Data Factory Pipeline Failure Due to Activity Error **Symptoms:** ADF monitoring dashboard reports a pipeline failure status, with the Databricks notebook activity showing a red cross and failure exit code. **Root Cause:** The underlying PySpark notebook encountered an unhandled exception (such as a bad file path, syntax error, or assertion failure) and exited with a non-zero status code. **Step-by-Step Fix:** 1. Click on the failed ADF activity run and follow the link to the Databricks cluster driver logs and",5e17c5da61aadfd4e26d9a7545c8dbb0
adf_pipeline_failed_activity_1,adf_pipeline_failed_activity,1,"failed ADF activity run and follow the link to the Databricks cluster driver logs and notebook output. 2. Inspect the stack trace to identify the exact line of code causing the failure. 3. Fix the notebook code, commit changes to git, and rerun the failed ADF pipeline from the failed activity. **Prevention:** Implement robust try-except error handling blocks within PySpark scripts and add custom logging to capture actionable error messages before notebook termination.",5e17c5da61aadfd4e26d9a7545c8dbb0
adf_trigger_missed_0,adf_trigger_missed,0,"Azure Data Factory Scheduled Trigger Missed Execution **Symptoms:** Expected daily ETL pipelines fail to kick off at their scheduled time, resulting in stale downstream analytical dashboards. **Root Cause:** The ADF tumbling window or schedule trigger was accidentally left in a `Stopped` state, or encountered transient platform throttling during scheduling evaluation. **Step-by-Step Fix:** 1. Navigate to the ADF Author/Monitor hub and inspect the trigger status. 2. Start the stopped trigger manually through the portal or Azure PowerShell SDK. 3. Use the",fdee35c336f699ba856dd7aa92d2f8fe
adf_trigger_missed_1,adf_trigger_missed,1,Start the stopped trigger manually through the portal or Azure PowerShell SDK. 3. Use the ADF rerun functionality to backfill missing window runs for the affected dates. **Prevention:** Set up Azure Monitor alerts for trigger failures and stopped states. Use CI/CD deployment pipelines to ensure triggers are automatically published in active states.,fdee35c336f699ba856dd7aa92d2f8fe
backfill_procedure_0,backfill_procedure,0,"Historical Backfill Procedure Overwriting Current Data **Symptoms:** Running a historical backfill pipeline inadvertently wipes out or corrupts the most recent production data partitions in the silver Delta table. **Root Cause:** The backfill script was executed without restricting date filters or used static partition overwrite mode, causing it to overwrite the entire table root instead of targeted historical date ranges. **Step-by-Step Fix:** 1. Access Delta table history using `DESCRIBE HISTORY table_name` to locate the version prior to the backfill execution. 2.",85506b9751170dbd4378c72f2cfd25d4


In [0]:
display(spark.sql(f"SHOW TBLPROPERTIES {TABLE_A}"))

key,value
delta.enableChangeDataFeed,true
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.changeDataFeed,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7
delta.parquet.compression.codec,zstd
delta.parquet.format.version,2.12.0
